# 🔬 Virtual Lab 4: Running OpenAI Models on LlamaIndex & LangChain  

<div style="border: 2px solid #4CAF50; padding: 15px; border-radius: 10px; background-color: #f4f4f4;">

### 🚀 **Platform**  
**OpenAI**  

### 🏷️ **Models Used**  
- **gpt-4o-mini**  
- **gpt-3.5-turbo**  

### 🛠️ **Frameworks Used**  
- **LlamaIndex**  
- **LangChain / LangGraph**  

</div>

In [ ]:
!pip install pypdf2

In [2]:
!pip show pypdf2

Name: PyPDF2
Version: 3.0.1
Summary: A pure-python PDF library capable of splitting, merging, cropping, and transforming PDF files
Home-page: 
Author: 
Author-email: Mathieu Fenniak <biziqe@mathieu.fenniak.net>
License: 
Location: C:\Workbench\LlamaIndex-LangChain-RAG\venv-LlamaIndex-LangChain\Lib\site-packages
Requires: 
Required-by: 


In [ ]:
!pip install langchain langchain-core openai

In [ ]:
!pip install -U langchain-community

In [ ]:
!pip install -U langchain langchain-core openai langchain-community

In [ ]:
!pip install -U langchain-openai langchain langchain-core langchain-community

In [1]:
import os
import openai
import requests
import zipfile
import sqlite3
import json
from sqlalchemy import create_engine, text
from pydantic import BaseModel
from PyPDF2 import PdfReader
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

### OpenAI API Setup & Configuration

In this section, we set up the OpenAI API client and define a function (`call_gpt`)  
to interact with **GPT-4o Mini**.

In [9]:
# Set OpenAI API Key
# os.environ['OPENAI_API_KEY'] = 'add-your-api-key'
# openai.api_key = os.getenv('OPENAI_API_KEY')

from os import getenv
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
openai_api_key = getenv('OPENAI_API_KEY')

In [10]:
# Initialize OpenAI client
client = openai.OpenAI(api_key=openai_api_key)

In [11]:
# Set global model configuration
llm_config = {'model': 'gpt-4o-mini'}

In [12]:
# Call complete with a prompt
def call_gpt(prompt):
    response = \
        client.chat.completions.create(
            model=llm_config['model'],
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.7,
        )
    return response.choices[0].message.content

### Document Download & Text Extraction

This section downloads PDF documents related to **Drake and Kendrick Lamar**,  
extracts their text content, and loads them for further processing.

- **Download PDFs**: Fetches the documents from Dropbox and saves them locally.  
- **Extract Text**: Reads the PDFs using `PyPDF2` and converts them into plain text.  
- **Load Documents**: Stores the extracted text in variables (`docs_kendrick`, `docs_drake`, `docs_both`)  
  for querying and analysis.

In [13]:
# Function to download PDFs
def download_file(url, filepath):
    response = requests.get(url, stream=True)
    with open(filepath, 'wb') as file:
        for chunk in response.iter_content(chunk_size=1024):
            file.write(chunk)

os.makedirs('data', exist_ok=True)

In [14]:
# Download documents
pdf_urls = {
    'drake_kendrick_beef': 'https://www.dropbox.com/scl/fi/t1soxfjdp0v44an6sdymd/drake_kendrick_beef.pdf?rlkey=u9546ymb7fj8lk2v64r6p5r5k&st=wjzzrgil&dl=1',
    'drake': 'https://www.dropbox.com/scl/fi/nts3n64s6kymner2jppd6/drake.pdf?rlkey=hksirpqwzlzqoejn55zemk6ld&st=mohyfyh4&dl=1',
    'kendrick': 'https://www.dropbox.com/scl/fi/8ax2vnoebhmy44bes2n1d/kendrick.pdf?rlkey=fhxvn94t5amdqcv9vshifd3hj&st=dxdtytn6&dl=1',
}

In [15]:
for name, url in pdf_urls.items():
    download_file(url, f'data/{name}.pdf')

In [16]:
# Function to extract text from PDFs
def extract_text_from_pdf(filepath):
    with open(filepath, 'rb') as file:
        reader = PdfReader(file)
        text = '\n\n'.join([page.extract_text() for page in reader.pages if page.extract_text()])
    return text

In [17]:
# Load documents
docs = {
    'drake_kendrick_beef': extract_text_from_pdf('data/drake_kendrick_beef.pdf'),
    'drake': extract_text_from_pdf('data/drake.pdf'),
    'kendrick': extract_text_from_pdf('data/kendrick.pdf'),
}

In [18]:
# Initialize OpenAI Embeddings and Vector Store
embedding_model = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = InMemoryVectorStore(embedding=embedding_model)

In [19]:
# Add documents to vector store 
for name, text in docs.items():
    doc = Document(page_content=text, metadata={'source': name})
    vector_store.add_documents([doc])  # Remove embedding_model from here

### Basic GPT-4o Mini Query & Streaming Response

This section demonstrates how to interact with **GPT-4o Mini** using both standard  
and streaming responses.

- **Basic Completion**: Calls `call_gpt()` to get a simple text-based response.  
- **Streaming Response**: Uses `stream_gpt()` to receive output in real-time,  
  printing the response incrementally as it's generated.  

In [20]:
response = call_gpt('Do you like Drake or Kendrick better?')
print(response)

As an AI, I don't have personal preferences or feelings. However, I can tell you that both Drake and Kendrick Lamar are highly influential artists in the hip-hop genre, each with their own unique style and contributions to music. Drake is known for his catchy hooks and emotional lyrics, while Kendrick is celebrated for his storytelling and social commentary. It often comes down to personal taste! Which one do you prefer?


In [22]:
# Streaming response
def stream_gpt(prompt):
    response = \
        client.chat.completions.create(
            model=llm_config['model'],
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.7,
            stream=True,
        )
    for chunk in response:
        if chunk.choices:
            print(chunk.choices[0].delta.content, end='')

stream_gpt("You're a Drake fan. Tell me why you like Drake more than Kendrick.")

As a Drake fan, I appreciate his versatility and ability to blend different genres seamlessly. His music often combines elements of hip-hop, R&B, and pop, making it accessible to a wide audience. Drake's storytelling is relatable, often reflecting on personal experiences, relationships, and success, which resonates with many listeners.

Additionally, his knack for catchy hooks and memorable melodies makes his songs easy to enjoy and sing along to. Drake's charisma and larger-than-life persona also contribute to his appeal, as he often engages with fans through social media and has a strong presence in the pop culture landscape.

While Kendrick Lamar is undoubtedly a brilliant artist with profound lyrical depth and social commentary, I find that Drake's style and sound speak to me personally, making him a standout in my music library. His ability to continuously evolve and stay relevant in the industry is also impressive, allowing him to maintain a strong connection with fans over the y

### Multi-Turn Chat with GPT-4o Mini

This section demonstrates **structured conversations** with GPT-4o Mini using a list of messages.

- **Role-Based Messages**: The model is assigned a **system role** (e.g., acting as Kendrick).  
- **User Interaction**: The user provides an input query, and GPT-4o Mini generates a response.    

In [23]:
# Call chat with a list of messages
def chat_with_gpt(messages):
    response = \
        client.chat.completions.create(
            model=llm_config['model'],
            messages=messages,
            temperature=0.7,
        )
    return response.choices[0].message.content

messages = [
    {'role': 'system', 'content': 'You are Kendrick.'},
    {'role': 'user', 'content': 'Write a verse.'},
]
response = chat_with_gpt(messages)

In [24]:
print(response)

Sure! Here’s a verse for you:

In the city lights, where dreams collide,  
Chasing shadows, where the truth can’t hide,  
Every step I take, a rhythm in my stride,  
Life’s a canvas, and I’m painting with my pride.  

Voices in my head, they whisper, “Keep it real,”  
Navigating through the pain, it’s a part of the deal,  
Rise from the ashes, like a phoenix, I reveal,  
Stronger than the struggle, yeah, that’s how I feel.  

Let me know if you need anything else!


### Basic RAG (Retrieval-Augmented Generation) - Vector Search

This section demonstrates **retrieving and answering questions** from documents  
using **GPT-4o Mini**.

- **Query-Based Search**: Uses `query_rag()` to fetch relevant information from the document.  
- **Contextual Responses**: The model is provided with document content to generate informed answers.   

In [25]:
def query_rag_with_embedding(prompt, top_k=3, max_tokens=3000):
    relevant_docs = vector_store.similarity_search(prompt, k=top_k)
    context = '\n\n'.join([doc.page_content[:max_tokens] for doc in relevant_docs])
    full_prompt = f'Using the following retrieved information, answer the question: {prompt}\n\n{context}'

    return call_gpt(full_prompt)

In [26]:
response = query_rag_with_embedding('Tell me about family matters')
print(response)

The retrieved information primarily discusses the ongoing feud between rappers Kendrick Lamar and Drake, highlighting its impact on hip-hop culture. The conflict has escalated recently, with both artists exchanging diss tracks filled with personal jabs and competitive energy. Kendrick Lamar initiated this latest chapter by making pointed references to Drake in his guest verse on the song "Like That," leading to a swift and intense response from Drake, who released a three-part reply accompanied by a music video.

This beef is significant in the context of hip-hop's evolving dynamics, illustrating how public rivalries can shape artists' legacies and influence the genre's rules of engagement. The exchange reflects a broader trend in rap where personal attacks and competitive spirit are increasingly prominent, marking a shift in how artists interact with one another.

Both Kendrick and Drake are influential figures in contemporary music, with Kendrick recognized for his social commentary 

### Basic RAG (Retrieval-Augmented Generation) - Summarization

This section demonstrates **summarizing document content** using **GPT-4o Mini**.

- **Context-Based Summarization**: Uses `summarize_rag()` to extract key insights from documents.  
- **Efficient Information Extraction**: The model condenses long-form content into a concise response.  

In [27]:
def query_rag_with_embedding(prompt, top_k=3, max_doc_length=1000):
    relevant_docs = vector_store.similarity_search(prompt, k=top_k)    
    truncated_docs = [doc.page_content[:max_doc_length] for doc in relevant_docs]
    context = '\n\n'.join(truncated_docs)
    
    full_prompt = f'Using the following retrieved information, answer the question: {prompt}\n\n{context}'
    
    response = call_gpt(full_prompt)
    
    return response

In [28]:
response = query_rag_with_embedding('Tell me about family matters')
print(response)

Family matters can encompass various aspects, including relationships, responsibilities, and dynamics within a family unit. In the context of Kendrick Lamar and Drake, their family backgrounds and personal lives play a significant role in shaping their artistry and public personas.

Kendrick Lamar Duckworth, born June 17, 1987, in Compton, California, often incorporates themes of family, community, and social issues into his music. His upbringing in a challenging environment has influenced his lyrics, leading to a focus on social commentary and personal narratives that resonate with many listeners. Kendrick's engagement with family matters can be seen in his songs, which frequently address the complexities of familial relationships and the impact of his background.

On the other hand, Aubrey Drake Graham, born October 24, 1986, in Toronto, Canada, also reflects on family in his work. Drake's familial connections, including his uncle Larry Graham, have played a role in his musical journ

### Advanced RAG (Routing & Sub-Questions)

This section implements an **intelligent query router** that determines whether  
to perform **vector search** or **summarization** based on the user's intent.

- **Automatic Routing**: GPT-4o Mini decides if a query requires **search** (fact retrieval)  
  or **summary** (document overview).  
- **Dynamic Query Processing**: The model selects the appropriate approach and generates a response.   

In [29]:
def query_router_with_embedding(prompt, top_k=3, max_tokens=3000):
    routing_prompt = \
        'Determine the best mode (search or summary) to process the given user query based on intent. ' + \
        "Return only 'search' if the query seeks specific facts, or 'summary' if the query requires summarization. " + \
        "Respond with only 'search' or 'summary'.\n\n" + \
        f'User Query: {prompt}'

    response = \
        client.chat.completions.create(
            model=llm_config['model'],
            messages=[{'role': 'user', 'content': routing_prompt}],
            temperature=0,
        )

    mode = response.choices[0].message.content.strip().lower()

    relevant_docs = vector_store.similarity_search(prompt, k=top_k)
    context = "\n\n".join([doc.page_content[:max_tokens] for doc in relevant_docs]) 

    if mode == 'search':
        full_prompt = f'Using the following retrieved information, find specific facts related to: {prompt}\n\n{context}'
    elif mode == 'summary':
        full_prompt = f'Summarize the document with respect to: {prompt}\n\n{context}'
    else:
        full_prompt = f'{prompt}\n\n{context}' 

    return call_gpt(full_prompt)

In [30]:
response_search = query_router_with_embedding("Tell me about the song 'Meet the Grahams' - why is it significant")
print(response_search)

The song "Meet the Grahams" is significant as part of the ongoing feud between Kendrick Lamar and Drake, marking a pivotal moment in the landscape of contemporary hip-hop. Released amidst heightened tensions, the song serves as a direct response to Drake's earlier tracks, continuing a pattern of back-and-forth diss exchanges that have captivated audiences and reshaped the rules of engagement in rap rivalries. 

Kendrick's aggressive tone in "Meet the Grahams" signals a departure from friendly competition to outright confrontation, with implications that could affect both artists' legacies. This new chapter in their rivalry showcases the intensity and artistry involved in hip-hop diss tracks, further elevating the stakes in an already competitive genre. The significance of the song lies not only in its lyrical content but also in its role as a catalyst for discussions about authenticity, rivalry, and the evolution of rap culture.


In [31]:
response_summary = query_router_with_embedding("Summarize the significance of 'Meet the Grahams'")
print(response_summary)

The document discusses the ongoing rap feud between Kendrick Lamar and Drake, highlighting its recent developments and cultural significance within the hip-hop community. Notable events include a series of diss tracks exchanged between the two artists, which escalated tensions and marked a new chapter in their rivalry. Kendrick's aggressive stance, declaring "choosing violence," signifies a shift in the rules of engagement in hip-hop, suggesting that the competition is more intense and personal than before.

Kendrick's recent verse in a collaboration with Future and Metro Boomin has been interpreted as a direct challenge to Drake, indicating that the rivalry has moved beyond mere competition to a more confrontational tone. The document emphasizes how this beef not only impacts the legacies of both artists but also reshapes the landscape of rap music. The exchange of diss tracks is characterized as potentially transformative for the genre, reflecting a cultural moment where personal att

### Break Complex Questions into Sub-Questions

This section implements a **Sub-Question Query Engine** that determines  
whether a query is related to **Drake or Kendrick Lamar** and retrieves  
relevant information accordingly.

- **Automatic Subject Classification**: GPT-4o Mini classifies the query as  
  related to **Drake** or **Kendrick** before fetching data.  
- **Targeted Query Execution**: Uses `docs_drake` if the query is about Drake  
  and `docs_kendrick` if it's about Kendrick.   

In [32]:
def determine_subject(prompt):
    classification_prompt = \
        "Determine whether the following question is about 'Drake' or 'Kendrick Lamar'. " + \
        "Return only 'drake' or 'kendrick'.\n\n" + \
        f'User Question: {prompt}'
    
    response = \
        client.chat.completions.create(
            model=llm_config['model'],
            messages=[{'role': 'user', 'content': classification_prompt}],
            temperature=0,
        )
    
    return response.choices[0].message.content.strip().lower()

In [33]:
def sub_question_query_engine_with_embedding(prompt, top_k=3, max_tokens=3000):
    subject = determine_subject(prompt)
    
    if subject == 'drake':
        relevant_docs = vector_store.similarity_search(prompt, k=top_k)
    else:  
        relevant_docs = vector_store.similarity_search(prompt, k=top_k)

    context = '\n\n'.join([doc.page_content[:max_tokens] for doc in relevant_docs])

    full_prompt = f'Using the following retrieved information, answer the question: {prompt}\n\n{context}'

    return call_gpt(full_prompt)

In [34]:
response = sub_question_query_engine_with_embedding('Which albums did Drake release in his career?')
print(response)

Drake has released the following studio albums in his career:

1. **Thank Me Later** (2010)
2. **Take Care** (2011)
3. **Nothing Was the Same** (2013)
4. **Views** (2016)
5. **Scorpion** (2018)
6. **Certified Lover Boy** (2021)
7. **Honestly, Nevermind** (2022)
8. **Her Loss** (2022, collaborative album with 21 Savage)
9. **For All the Dogs** (2023) 

In addition to these albums, he started his career with several mixtapes, including "Room for Improvement" (2006), "Comeback Season" (2007), and "So Far Gone" (2009).


### Text-to-SQL with GPT-4o Mini

This section demonstrates **converting natural language queries into SQL**  
to retrieve data from an SQLite database.

- **Database Setup**:  
  - Downloads and extracts the **Chinook SQLite database**, which contains  
    music-related tables like `albums`, `artists`, and `tracks`.  
  - Initializes a connection to `chinook.db` using SQLAlchemy.  

- **SQL Query Generation**:  
  - Uses GPT-4o Mini to **convert natural language questions into SQL queries**.  
  - Restricts queries to the tables: `albums`, `artists`, and `tracks`.  

- **Query Execution**:  
  - Runs the generated SQL queries on the database and retrieves the results.  

This setup allows **seamless querying of structured data** using natural language. 🚀

In [35]:
def download_file(url, filepath):
    response = requests.get(url, stream=True)
    with open(filepath, 'wb') as file:
        for chunk in response.iter_content(chunk_size=1024):
            file.write(chunk)

# Create data directory
os.makedirs('data', exist_ok=True)

In [36]:
download_file('https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip', 'data/chinook.zip')
with zipfile.ZipFile('data/chinook.zip', 'r') as zip_ref:
    zip_ref.extractall('data/')

In [37]:
engine = create_engine('sqlite:///data/chinook.db')

In [38]:
tables_schema = {
    'albums': 'AlbumId, Title, ArtistId',
    'artists': 'ArtistId, Name',
    'tracks': 'TrackId, Name, AlbumId, Composer, MediaTypeId, GenreId, Milliseconds, Bytes, UnitPrice',
}

In [39]:
schema_docs = [
    Document(page_content=f'Table: {table}\nColumns: {columns}', metadata={'table': table})
    for table, columns in tables_schema.items()
]

In [40]:
vector_store.add_documents(schema_docs)

['ba00173f-6922-4055-bbd1-35a4788711aa',
 '7ffb43e8-4773-48fc-858d-251334b4ebe5',
 'd879f4bb-f3cf-46a9-a7da-e9cc5c4ab2c4']

In [41]:
def retrieve_relevant_schema(prompt, top_k=2):
    relevant_schema = vector_store.similarity_search(prompt, k=top_k)
    return '\n\n'.join([doc.page_content for doc in relevant_schema])

In [42]:
from textwrap import dedent

def generate_sql_query_with_embedding(natural_language_query):
    relevant_schema = retrieve_relevant_schema(natural_language_query)

    prompt = \
        dedent(
            f"""\
            Convert the following natural language question into a SQL query for a SQLite database.
            Use only the relevant schema details provided below:
        
            {relevant_schema}
        
            **Now generate a SQL query for the following request:**
            
            "{natural_language_query}"
            
            **Return only the SQL query, without any explanation.**""",
        )

    response = \
        client.chat.completions.create(
            model=llm_config['model'],
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0,
        )

    sql_query = response.choices[0].message.content.strip()

    if '```sql' in sql_query:
        sql_query = sql_query.split('```sql')[1].split('```')[0].strip()

    return sql_query

In [43]:
from sqlalchemy.sql import text  

def query_sql_database_with_embedding(natural_language_query):
    sql_query = generate_sql_query_with_embedding(natural_language_query)
    print(f"Generated SQL Query: {sql_query}")
    with engine.connect() as connection:
        result = connection.execute(text(sql_query))  
        return [row for row in result]

In [44]:
response_albums = query_sql_database_with_embedding('What are some albums?')
print('Albums:', response_albums)

Generated SQL Query: SELECT Title FROM albums;
Albums: [('For Those About To Rock We Salute You',), ('Balls to the Wall',), ('Restless and Wild',), ('Let There Be Rock',), ('Big Ones',), ('Jagged Little Pill',), ('Facelift',), ('Warner 25 Anos',), ('Plays Metallica By Four Cellos',), ('Audioslave',), ('Out Of Exile',), ('BackBeat Soundtrack',), ('The Best Of Billy Cobham',), ('Alcohol Fueled Brewtality Live! [Disc 1]',), ('Alcohol Fueled Brewtality Live! [Disc 2]',), ('Black Sabbath',), ('Black Sabbath Vol. 4 (Remaster)',), ('Body Count',), ('Chemical Wedding',), ('The Best Of Buddy Guy - The Millenium Collection',), ('Prenda Minha',), ('Sozinho Remix Ao Vivo',), ('Minha Historia',), ('Afrociberdelia',), ('Da Lama Ao Caos',), ('Acústico MTV [Live]',), ('Cidade Negra - Hits',), ('Na Pista',), ('Axé Bahia 2001',), ('BBC Sessions [Disc 1] [Live]',), ('Bongo Fury',), ('Carnaval 2001',), ('Chill: Brazil (Disc 1)',), ('Chill: Brazil (Disc 2)',), ('Garage Inc. (Disc 1)',), ('Greatest Hits II'

In [45]:
response_artists = query_sql_database_with_embedding('What are some artists? Limit it to 5.')
print('Artists:', response_artists)

Generated SQL Query: SELECT * FROM artists LIMIT 5;
Artists: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


In [46]:
response_tracks = query_sql_database_with_embedding('What are some tracks from the artist AC/DC? Limit it to 3')
print('AC/DC Tracks:', response_tracks)

Generated SQL Query: SELECT Name FROM tracks WHERE Composer = 'AC/DC' LIMIT 3;
AC/DC Tracks: [('Go Down',), ('Dog Eat Dog',), ('Let There Be Rock',)]


### Structured Data Extraction using GPT-4o Mini

This section demonstrates **extracting structured data** from natural language  
using **GPT-4o Mini** and returning it in **JSON format**.

- **Data Extraction Process**:  
  - The model generates structured data for a **restaurant** in a given city.  
  - The output must be a **valid JSON object** containing:  
    - `name`: The restaurant's name.  
    - `city`: The specified city.  
    - `cuisine`: The type of cuisine served.  

In [47]:
from langchain_core.documents import Document
from pydantic import BaseModel

class Restaurant(BaseModel):
    """A restaurant with name, city, and cuisine."""
    name: str
    city: str
    cuisine: str

In [49]:
restaurant_data = [
    {'name': "Joe's Seafood", 'city': 'Miami', 'cuisine': 'Seafood'},
    {'name': 'Pasta Paradise', 'city': 'New York', 'cuisine': 'Italian'},
    {'name': 'Sushi Haven', 'city': 'San Francisco', 'cuisine': 'Japanese'},
    {'name': 'BBQ King', 'city': 'Austin', 'cuisine': 'BBQ'},
]

In [50]:
restaurant_docs = [
    Document(page_content=f"Restaurant: {r['name']}, City: {r['city']}, Cuisine: {r['cuisine']}", metadata={'city': r['city']})
    for r in restaurant_data
]

vector_store.add_documents(restaurant_docs)

['483e2d3e-b7ba-48ae-81a1-569989bd8926',
 '653fe90c-5020-4e7d-a086-372988dcacb3',
 '3948594e-02d8-4f75-853d-9092b019eafe',
 '28a7d818-87d2-48eb-a105-16267db96f67']

In [51]:
def retrieve_restaurants_by_city(city_name, top_k=3):
    relevant_docs = vector_store.similarity_search(city_name, k=top_k)
    return relevant_docs

In [52]:
def extract_structured_data_with_embedding(city_name):
    relevant_restaurants = retrieve_restaurants_by_city(city_name)

    if not relevant_restaurants:
        prompt = f'Generate a restaurant in a given city: {city_name}. Return only a valid JSON object with keys: name, city, cuisine, without markdown formatting.'
        response = client.chat.completions.create(
            model=llm_config['model'],
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0
        )

        structured_data = response.choices[0].message.content.strip()

        if '```json' in structured_data:
            structured_data = structured_data.split('```json')[1].split('```')[0].strip()

        try:
            json_data = json.loads(structured_data)
            return Restaurant.model_validate(json_data)
        except json.JSONDecodeError as e:
            print('JSON Decode Error:', e)
            print('Raw Response:', structured_data)
            return None

    restaurant_list = []
    for doc in relevant_restaurants:
        parts = doc.page_content.split(", ")
        restaurant_dict = {
            'name': parts[0].split(": ")[1],
            'city': parts[1].split(": ")[1],
            'cuisine': parts[2].split(": ")[1],
        }
        restaurant_list.append(Restaurant(**restaurant_dict))

    return restaurant_list[0] if restaurant_list else None  # Return the first matching restaurant

In [53]:
restaurant_obj = extract_structured_data_with_embedding('Miami')
print(restaurant_obj)

name="Joe's Seafood" city='Miami' cuisine='Seafood'


### Adding Chat History to RAG (Chat Engine)

This section implements a **stateful chatbot** that integrates **chat history**  
with **Retrieval-Augmented Generation (RAG)** to provide more context-aware responses.

- **Chat Memory Management**:  
  - Stores past interactions in `ChatMemory` to maintain conversation flow.  
  - Limits stored messages to prevent exceeding token constraints.  

- **Contextual Retrieval**:  
  - Combines **user input, past chat history, and relevant document context**  
    (e.g., about Kendrick & Drake) to generate informed responses.  

In [54]:
from langchain_core.documents import Document

class ChatMemoryWithEmbeddings:
    def __init__(self, token_limit=10000): 
        self.token_limit = token_limit
        self.messages = []
        self.vector_store = vector_store  

    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})

        doc = Document(page_content=content[:1000], metadata={"role": role}) 
        self.vector_store.add_documents([doc])

        if len(self.messages) > 20:  
            self.messages.pop(0)

    def retrieve_relevant_history(self, prompt, top_k=2):
        relevant_docs = self.vector_store.similarity_search(prompt, k=top_k)
        return '\n\n'.join([doc.page_content[:500] for doc in relevant_docs])  

In [55]:
memory = ChatMemoryWithEmbeddings()

In [56]:
def chat_with_history_using_embeddings(prompt):
    relevant_history = memory.retrieve_relevant_history(prompt, top_k=2)

    relevant_docs = vector_store.similarity_search(prompt, k=2)
    document_context = '\n\n'.join([doc.page_content[:1000] for doc in relevant_docs])  # Limit document size

    context_prompt = \
        'You are a chatbot, able to have normal interactions, as well as talk ' + \
        'about the Kendrick and Drake beef. Use the retrieved chat history and document context:\n\n' + \
        f'Chat History (trimmed):\n{relevant_history}\n\n' + \
        f'Relevant Documents (trimmed):\n{document_context}\n\n' + \
        'Instruction: Use the previous chat history, or the context above, to interact and help the user.'

    messages = [
        {'role': 'system', 'content': context_prompt},
        {'role': 'user', 'content': prompt},
    ]

    total_tokens = sum(len(msg['content'].split()) for msg in messages)
    if total_tokens > 10000:
        print('Warning: Trimming messages to fit within token limit.')
        messages = messages[-5:]  

    response = \
        client.chat.completions.create(
            model=llm_config['model'],
            messages=messages,
            temperature=0.7,
        )

    response_content = response.choices[0].message.content.strip()
    memory.add_message('assistant', response_content) 

    return response_content

In [57]:
response = chat_with_history_using_embeddings('Tell me about the songs Drake released in the beef.')
print(response)

In the recent escalation of the beef between Drake and Kendrick Lamar, Drake released a three-part response track that took aim at Kendrick and others involved in the ongoing feud. This release came on the first weekend of May 2024. The tracks were accompanied by a music video, showcasing Drake's signature style while addressing the diss and tension between the two artists.

While the specifics of the tracks' titles and lyrical content may not be detailed, they were described as densely packed with personal shots and power punches aimed at Kendrick and others in the rap scene. This release marked a significant moment in their rivalry, showcasing not just lyrical skills but also strategic timing in the ongoing back-and-forth between the two hip-hop giants.

If you're interested in specific lyrics or themes from those songs, I can help with that!


In [58]:
response = chat_with_history_using_embeddings('What about Kendrick?')
print(response)

Kendrick Lamar, born Kendrick Lamar Duckworth on June 17, 1987, is an acclaimed American rapper, songwriter, and filmmaker from Compton, California. He gained recognition for his impactful lyrics that often include social commentary and political criticism, which has influenced a rise in social consciousness among his listeners. Kendrick is known for his unique storytelling ability and has received numerous accolades, including the Pulitzer Prize for Music, making him the only non-classical or jazz musician to receive this honor.

He started releasing music under the name K.Dot during his high school years and signed with Top Dawg Entertainment in 2005, where he rose to prominence. His albums, like "good kid, m.A.A.d city," "To Pimp a Butterfly," and "DAMN.," have been critically acclaimed and are considered modern classics.

If you're interested in his ongoing beef with Drake, there's been a lot of back-and-forth recently, with both artists trading diss tracks and jabs in their music.

## 7. Agents

Here we build agents with gpt-4o-mini . We perform RAG over simple functions as well as the documents above.

In [59]:
import nest_asyncio
import json
from langchain_core.documents import Document

nest_asyncio.apply()

In [60]:
# Define mathematical functions
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

def add(a: int, b: int) -> int:
    """Add two integers and return the result."""
    return a + b

def subtract(a: int, b: int) -> int:
    """Subtract two integers and return the result."""
    return a - b

def divide(a: int, b: int) -> int:
    """Divide two integers and return the result."""
    return a / b if b != 0 else 'Cannot divide by zero'

# Function map for the agent
tools = {
    'multiply': multiply,
    'add': add,
    'subtract': subtract,
    'divide': divide,
}

In [61]:
function_examples = [
    {'function': 'multiply', 'query': 'What is 5 times 3?', 'args': [5, 3]},
    {'function': 'add', 'query': 'What is 10 plus 4?', 'args': [10, 4]},
    {'function': 'subtract', 'query': 'What is 20 minus 7?', 'args': [20, 7]},
    {'function': 'divide', 'query': 'What is 15 divided by 5?', 'args': [15, 5]},
]

In [62]:
function_docs = [
    Document(page_content=f"Query: {ex['query']}, Function: {ex['function']}, Args: {ex['args']}")
    for ex in function_examples
]

vector_store.add_documents(function_docs)

['6e8d76eb-c4d2-4851-8843-0ab85db0702a',
 '4149add4-26db-4ff2-b641-d6f99696aa1a',
 '6334a268-6a82-4b73-939a-603d65176975',
 '4ba4640d-a8a2-43c3-ad83-b072c2fa8431']

In [63]:
def retrieve_relevant_function_examples(prompt, top_k=2):
    relevant_docs = vector_store.similarity_search(prompt, k=top_k)
    return '\n\n'.join([doc.page_content for doc in relevant_docs])

In [64]:
def execute_tool(tool_name, *args):
    if tool_name in tools:
        return tools[tool_name](*args)
    return 'Invalid tool request'

In [65]:
def agent_chat_with_embeddings(prompt):
    relevant_examples = retrieve_relevant_function_examples(prompt)

    system_prompt = \
        'You are a smart assistant capable of performing arithmetic operations. ' + \
        'You can use the following functions: multiply, add, subtract, divide. ' + \
        'Below are relevant past function calls for reference:\n\n' + \
        f'{relevant_examples}\n\n' + \
        'When given a math question, return the correct function and inputs in JSON format. ' + \
        'Ensure the JSON output follows this format:\n\n' + \
        '{\n  \"function\": \"function_name\",\n  \"arguments\": [arg1, arg2]\n}\n\n' + \
        'Return only valid JSON with no extra text or formatting.'

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': prompt},
    ]

    response = \
        client.chat.completions.create(
            model=llm_config['model'],
            messages=messages,
            temperature=0,
        )

    response_content = response.choices[0].message.content.strip()

    try:
        if '```json' in response_content:
            response_content = response_content.split('```json')[1].split('```')[0].strip()

        tool_request = json.loads(response_content)
        tool_name = tool_request.get('function')
        arguments = tool_request.get('arguments', [])

        result = execute_tool(tool_name, *arguments)
        return result
    except json.JSONDecodeError:
        print('Failed to parse response:', response_content)
        return 'Error: Could not parse the response as JSON.'

In [66]:
response = agent_chat_with_embeddings('What is (121 + 2) * 5?')
print(response)

615


### ReAct Agent with RAG QueryEngine Tools

This section implements a **ReAct-style agent** that dynamically selects between  
**Drake-related** and **Kendrick-related** document retrieval using **GPT-4o Mini**.

- **ReAct Framework**:  
  - Uses **reasoning + action** to **select the right tool** for querying.  
  - Determines whether to call **`query_drake`** or **`query_kendrick`** based on the prompt.  

- **Tool-Based Query Execution**:  
  - **GPT-4o Mini** analyzes the query and responds in **JSON format** specifying the correct tool.  
  - The tool is then **executed dynamically** to fetch relevant information.  

This approach enables **intelligent document selection** and **enhanced retrieval accuracy**. 🚀

In [67]:
import json
from langchain_core.documents import Document

def query_rag_with_embedding(prompt, top_k=3, max_tokens=3000):
    relevant_docs = vector_store.similarity_search(prompt, k=top_k)

    context = '\n\n'.join([doc.page_content[:max_tokens] for doc in relevant_docs])

    full_prompt = f'Using the following retrieved information, answer the question: {prompt}\n\n{context}'

    return call_gpt(full_prompt)

In [68]:
def determine_subject_using_embeddings(prompt):
    classification_prompt = \
        "Determine whether the following question is about 'Drake' or 'Kendrick Lamar'. " + \
        "Return only 'drake' or 'kendrick'.\n\n" + \
        f'User Query: {prompt}'
    
    response = \
        client.chat.completions.create(
            model=llm_config['model'],
            messages=[{'role': 'user', 'content': classification_prompt}],
            temperature=0,
        )
    
    return response.choices[0].message.content.strip().lower()

In [70]:
def react_agent_with_embeddings(prompt):
    subject = determine_subject_using_embeddings(prompt)

    relevant_docs = vector_store.similarity_search(prompt, k=3)
    document_context = "\n\n".join([doc.page_content[:1000] for doc in relevant_docs])  # Limit document size

    system_prompt = \
        'You are an AI assistant capable of retrieving and summarizing information about Drake and Kendrick Lamar. ' + \
        'Use the retrieved document context below to generate your response:\n\n' + \
        f'Relevant Documents (trimmed):\n{document_context}\n\n' + \
        "Instruction: Answer the user's query using the provided context."

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': prompt},
    ]

    response = \
        client.chat.completions.create(
            model=llm_config['model'],
            messages=messages,
            temperature=0,
        )

    return response.choices[0].message.content.strip()

In [71]:
response = react_agent_with_embeddings('Tell me about how Kendrick and Drake grew up')
print(response)

Kendrick Lamar, born Kendrick Lamar Duckworth on June 17, 1987, grew up in Compton, California. His upbringing in a city known for its struggles with crime and poverty heavily influenced his music and lyrical content. Kendrick began releasing music under the name K.Dot during his high school years and signed with Top Dawg Entertainment in 2005, where he started to gain recognition for his impactful lyrics that often include social commentary and political criticism.

On the other hand, the context does not provide specific details about Drake's upbringing. However, it is known that Drake, born Aubrey Drake Graham on October 24, 1986, was raised in Toronto, Canada. He experienced a different environment compared to Kendrick, growing up in a mixed-race household with a Jewish mother and an African American father. Drake's early career began in acting on the television series "Degrassi: The Next Generation" before he transitioned into music.

While both artists have different backgrounds 